# Phase 2A — The Collapse Curve
**Split regimes × core models.** Requires Phase 1 outputs in `$IKS_PROC_ROOT` (default `./processed`).

| Regime | Pool | Split unit | Isolates (vs previous rung) |
|---|---|---|---|
| R1a | full 385k (83% overlap) | random segment | — (the literature's protocol) |
| R1b | stride-6 (~64k, no window overlap) | random segment | duplicate-audio leakage |
| R2 | stride-6 | song-disjoint (frozen folds) | same-song leakage |
| R3 | stride-6 | artist-disjoint (frozen folds) | **the singer** |
| R4 | = R3 | + mean-logit vote per recording | segment-count inflation |

R4 needs **no extra training** — it is R3 re-scored at song level.
Grid = 4 regimes × 2 models (ResNet-18, CRNN) × 5 folds = **40 runs**, fully **resumable**
(each finished run writes `results2A/<name>.json` and is skipped on re-run).

**Recommended first:** set `IKS_SMOKE=1` below and run all cells once (~5–10 min) to verify the
pipeline end-to-end on your data, then delete the smoke results, set it back to `0` and restart & run all.
The script equivalent is `python scripts/phase2a_run_grid.py [--smoke]`.


In [ ]:
# ============ CELL 0: smoke-test switch ============
import os
os.environ['IKS_SMOKE'] = '0'   # '1' = quick pipeline check (1 fold/regime, 2 epochs); '0' = full grid

In [ ]:
# ============ CELL 1: Setup — load index + entire dataset into RAM ============
import os, gc, glob, json, time, shutil, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

if os.path.basename(os.getcwd()) == 'notebooks': os.chdir('..')   # resolve paths from the repo root
PROC_ROOT  = os.environ.get('IKS_PROC_ROOT', './processed')
RESULTS    = os.environ.get('IKS_RES2A', f"{PROC_ROOT}/results2A")
os.makedirs(RESULTS, exist_ok=True)
SMOKE = os.environ.get('IKS_SMOKE', '0') == '1'

# ---- master index ----
idxdf = pd.read_parquet(f"{PROC_ROOT}/index_with_folds.parquet")
idxdf = idxdf.sort_values('global_idx').reset_index(drop=True)
assert (idxdf['global_idx'].values == np.arange(len(idxdf))).all(), 'global_idx must be 0..N-1'
idxdf = idxdf.set_index('global_idx', drop=False)
for c in ['genre_id','artist_id','state_id','gender_id','fold_artist','fold_song']:
    idxdf[c] = idxdf[c].astype(int)
N_TOTAL = len(idxdf); print(f"index rows: {N_TOTAL:,}")
ID2GENRE = idxdf.drop_duplicates('genre_id').set_index('genre_id')['genre'].to_dict()
N_CLASSES = idxdf['genre_id'].nunique()

# ---- assemble X_ALL in RAM (float16; row i corresponds to global_idx i) ----
blocks = idxdf.groupby('style')['global_idx'].agg(['min','max','count']).sort_values('min')
for s, r in blocks.iterrows():
    assert r['max'] - r['min'] + 1 == r['count'], f"{s}: non-contiguous block"
X_ALL = np.empty((N_TOTAL, 128, 130), dtype=np.float16)
t0 = time.time()
for s, r in blocks.iterrows():
    Xs = np.load(f"{PROC_ROOT}/X_{s}.npy", mmap_mode='r')
    assert Xs.shape == (r['count'], 128, 130), f"{s}: shape {Xs.shape} != index"
    X_ALL[r['min']:r['max']+1] = Xs[:]
    del Xs
print(f"X_ALL in RAM: {X_ALL.nbytes/1e9:.2f} GB   (load {time.time()-t0:.0f}s)")

# ---- pools: full (overlapping) and stride-6 (non-overlapping, k*=22 -> ceil(130/22)=6) ----
STRIDE = 6
idxdf['seg_rank'] = idxdf.groupby('source').cumcount()
stride_pool = idxdf.loc[idxdf['seg_rank'] % STRIDE == 0, 'global_idx'].values
full_pool   = idxdf['global_idx'].values
print(f"full pool: {len(full_pool):,}   stride-{STRIDE} pool: {len(stride_pool):,}")
print(f"classes: {N_CLASSES} | SMOKE={SMOKE}")

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '(CPU)')

In [ ]:
# ============ CELL 2: Models, training loop, metrics ============
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score

def normalize_batch(x):          # dB in [-80, 0]  ->  [-1, 1]
    return (x + 40.0) / 40.0

class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.b2 = nn.BatchNorm2d(cout)
        self.down = None
        if stride != 1 or cin != cout:
            self.down = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        idn = x if self.down is None else self.down(x)
        h = F.relu(self.b1(self.c1(x)))
        h = self.b2(self.c2(h))
        return F.relu(h + idn)

class ResNet18(nn.Module):       # standard ResNet-18, 1-channel stem (no torchvision dependency)
    def __init__(self, n_classes):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 64, 7, 2, 3, bias=False), nn.BatchNorm2d(64),
                                  nn.ReLU(inplace=True), nn.MaxPool2d(3, 2, 1))
        def layer(cin, cout, s): return nn.Sequential(BasicBlock(cin, cout, s), BasicBlock(cout, cout))
        self.l1 = layer(64, 64, 1);  self.l2 = layer(64, 128, 2)
        self.l3 = layer(128, 256, 2); self.l4 = layer(256, 512, 2)
        self.fc = nn.Linear(512, n_classes)
    def forward(self, x):
        h = self.stem(x); h = self.l4(self.l3(self.l2(self.l1(h))))
        return self.fc(F.adaptive_avg_pool2d(h, 1).flatten(1))

class CRNN(nn.Module):           # 4 conv blocks -> BiGRU over time -> mean pool
    def __init__(self, n_classes):
        super().__init__()
        def blk(cin, cout): return nn.Sequential(nn.Conv2d(cin, cout, 3, 1, 1), nn.BatchNorm2d(cout),
                                                 nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.conv = nn.Sequential(blk(1, 32), blk(32, 64), blk(64, 128), blk(128, 128))
        self.gru = nn.GRU(128 * 8, 128, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(256, n_classes)
    def forward(self, x):                       # (B,1,128,130)
        h = self.conv(x)                        # (B,128,8,8)  [freq 128/16=8, time 130//16=8]
        h = h.permute(0, 3, 1, 2).flatten(2)    # (B, T=8, 128*8)
        out, _ = self.gru(h)
        return self.fc(out.mean(1))

def make_model(name, n_classes):
    return {'resnet18': ResNet18, 'crnn': CRNN}[name](n_classes)

def metric_pack(y, p, prefix):
    labels = np.unique(y)
    return {f'{prefix}_acc': float(accuracy_score(y, p)),
            f'{prefix}_macro_f1': float(f1_score(y, p, labels=labels, average='macro', zero_division=0))}

@torch.no_grad()
def predict(model, gidx, batch=1024):
    model.eval(); outs = []
    for i in range(0, len(gidx), batch):
        xb = torch.from_numpy(X_ALL[gidx[i:i+batch]].astype(np.float32)).unsqueeze(1)
        xb = normalize_batch(xb).to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE.type, enabled=(DEVICE.type == 'cuda')):
            outs.append(model(xb).float().cpu())
    return torch.cat(outs).numpy()

def song_level(gidx, logits):
    src = idxdf.loc[gidx, 'source'].values
    y   = idxdf.loc[gidx, 'genre_id'].values
    L = pd.DataFrame(logits); L['source'] = src
    agg = L.groupby('source').mean()
    ytrue = pd.Series(y, index=src).groupby(level=0).first().loc[agg.index].values
    return ytrue, agg.values.argmax(1)

def train_one_run(model_name, tr, va, te, epochs, batch=512, lr=3e-4, seed=0, log_every=10):
    torch.manual_seed(seed); np.random.seed(seed)
    model = make_model(model_name, N_CLASSES).to(DEVICE)
    ytr = idxdf.loc[tr, 'genre_id'].values
    counts = np.bincount(ytr, minlength=N_CLASSES).astype(np.float64)
    w = np.where(counts > 0, counts.sum() / (N_CLASSES * np.maximum(counts, 1)), 0.0)
    crit  = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=DEVICE))
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
    rng = np.random.default_rng(seed)
    best_f1, best_state, best_ep = -1.0, None, -1
    t0 = time.time()
    for ep in range(epochs):
        model.train(); perm = rng.permutation(tr)
        for i in range(0, len(perm), batch):
            gb = perm[i:i+batch]
            xb = torch.from_numpy(X_ALL[gb].astype(np.float32)).unsqueeze(1)
            xb = normalize_batch(xb).to(DEVICE, non_blocking=True)
            yb = torch.from_numpy(idxdf.loc[gb, 'genre_id'].values).to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                loss = crit(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        lv = predict(model, va); yv = idxdf.loc[va, 'genre_id'].values
        f1v = f1_score(yv, lv.argmax(1), labels=np.unique(yv), average='macro', zero_division=0)
        if f1v > best_f1:
            best_f1, best_ep = float(f1v), ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if ep % log_every == 0 or ep == epochs - 1:
            print(f"    ep {ep:>3}  val macroF1={f1v:.4f}  ({time.time()-t0:.0f}s)", flush=True)
    model.load_state_dict(best_state)
    logits = predict(model, te)
    y = idxdf.loc[te, 'genre_id'].values
    m = metric_pack(y, logits.argmax(1), 'seg')
    ys, ps = song_level(te, logits)
    m.update(metric_pack(ys, ps, 'song'))
    m.update({'best_epoch': best_ep, 'val_macro_f1': best_f1,
              'train_sec': round(time.time() - t0, 1), 'n_train': int(len(tr)), 'n_test': int(len(te))})
    return model, logits, m

In [ ]:
# ============ CELL 3: Run the grid (resumable) ============
def build_indices(regime, fold):
    if regime in ('R1a', 'R1b'):
        pool = full_pool if regime == 'R1a' else stride_pool
        perm = np.random.default_rng(100).permutation(pool)   # one fixed permutation -> disjoint test chunks
        chunks = np.array_split(perm, 5)
        te = chunks[fold]
        tr = np.concatenate([c for j, c in enumerate(chunks) if j != fold])
    else:
        col = 'fold_song' if regime == 'R2' else 'fold_artist'
        pool = stride_pool
        f = idxdf.loc[pool, col].values
        te, tr = pool[f == fold], pool[f != fold]
    return tr, te

def carve_val(tr, regime, fold, frac=0.10):
    rng = np.random.default_rng(1000 + fold)
    if regime in ('R1a', 'R1b'):
        perm = rng.permutation(tr); nv = int(len(tr) * frac)
        return perm[nv:], perm[:nv]
    key = 'source' if regime == 'R2' else 'artist_id'         # group-aware validation
    g = idxdf.loc[tr, key].values
    ug = rng.permutation(np.unique(g)); nv = max(1, int(round(len(ug) * frac)))
    vmask = np.isin(g, ug[:nv])
    return tr[~vmask], tr[vmask]

EPOCHS  = {'R1a': 20, 'R1b': 30, 'R2': 30, 'R3': 30}
MODELS  = ['resnet18', 'crnn']
REGIMES = ['R1a', 'R1b', 'R2', 'R3']
FOLDS   = range(5)
BATCH, LR = 512, 3e-4
if SMOKE:
    EPOCHS = {k: 2 for k in EPOCHS}; FOLDS = range(1); BATCH = 256

runs = [dict(regime=r, model=m, fold=f) for r in REGIMES for m in MODELS for f in FOLDS]
print(f"{len(runs)} runs queued -> {RESULTS}")

durs = []
for k, run in enumerate(runs):
    name = f"{run['regime']}_{run['model']}_fold{run['fold']}"
    jpath = f"{RESULTS}/{name}.json"
    if os.path.exists(jpath):
        print(f"[skip] {name} (done)"); continue
    tr, te = build_indices(run['regime'], run['fold'])
    if SMOKE: tr, te = tr[:4000], te[:1500]
    tr, va = carve_val(tr, run['regime'], run['fold'])
    print(f"[{k+1}/{len(runs)}] {name}  train={len(tr):,} val={len(va):,} test={len(te):,}", flush=True)
    model, logits, m = train_one_run(run['model'], tr, va, te,
                                     epochs=EPOCHS[run['regime']], batch=BATCH, lr=LR, seed=run['fold'])
    np.savez_compressed(f"{RESULTS}/{name}_preds.npz",
                        gidx=te, y=idxdf.loc[te, 'genre_id'].values, logits=logits.astype(np.float32))
    torch.save(model.state_dict(), f"{RESULTS}/{name}.pt")
    m.update(run); json.dump(m, open(jpath, 'w'), indent=1)
    durs.append(m['train_sec'])
    left = sum(1 for r2 in runs[k+1:] if not os.path.exists(f"{RESULTS}/{r2['regime']}_{r2['model']}_fold{r2['fold']}.json"))
    print(f"    seg acc={m['seg_acc']:.4f} F1={m['seg_macro_f1']:.4f} | song acc={m['song_acc']:.4f} "
          f"F1={m['song_macro_f1']:.4f} | {m['train_sec']:.0f}s | ETA ~{np.mean(durs)*left/60:.0f} min", flush=True)
    del model; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print("grid complete ✅")

In [ ]:
# ============ CELL 4: Collapse curve + tables ============
res = [json.load(open(p)) for p in sorted(glob.glob(f"{RESULTS}/R*_fold*.json"))]
R = pd.DataFrame(res); print(f"{len(R)} completed runs")

# R4 = the R3 runs re-scored at SONG level (no extra training)
rows = []
for _, r in R.iterrows():
    rows.append(dict(regime=r['regime'], model=r['model'], fold=r['fold'],
                     acc=r['seg_acc'], f1=r['seg_macro_f1']))
    if r['regime'] == 'R3':
        rows.append(dict(regime='R4', model=r['model'], fold=r['fold'],
                         acc=r['song_acc'], f1=r['song_macro_f1']))
C = pd.DataFrame(rows)
order = ['R1a', 'R1b', 'R2', 'R3', 'R4']
summary = (C.groupby(['regime', 'model'])
             .agg(acc_mean=('acc','mean'), acc_std=('acc','std'),
                  f1_mean=('f1','mean'),  f1_std=('f1','std'), n=('acc','size'))
             .reindex(pd.MultiIndex.from_product([order, MODELS], names=['regime','model'])))
print(summary.round(4).to_string())
summary.to_csv(f"{RESULTS}/summary_collapse.csv")

plt.figure(figsize=(8.5, 5))
xs = np.arange(len(order))
for m, c in zip(MODELS, ['#d62728', '#1f77b4']):
    mu = np.array([summary.loc[(r, m), 'acc_mean'] for r in order], dtype=float)
    sd = np.nan_to_num(np.array([summary.loc[(r, m), 'acc_std'] for r in order], dtype=float))
    plt.errorbar(xs, mu, yerr=sd, marker='o', capsize=4, label=m, color=c)
plt.xticks(xs, ['R1a\nrandom seg\n(full, 83% ovl)', 'R1b\nrandom seg\n(stride-6)',
                'R2\nsong-\ndisjoint', 'R3\nartist-\ndisjoint', 'R4\nartist-disj.\n+ song vote'])
plt.ylabel('accuracy'); plt.ylim(0, 1.0)
plt.axhline(1/15, ls=':', c='gray'); plt.text(0.05, 1/15 + .012, 'chance (1/15)', color='gray', fontsize=8)
plt.title('The Collapse Curve — what folk-genre accuracy is made of')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(f"{RESULTS}/fig_collapse_curve.png", dpi=150); plt.show()

# per-genre song-level accuracy under artist-disjoint eval (folds where the genre appears)
per = {}
for p in sorted(glob.glob(f"{RESULTS}/R3_resnet18_fold*_preds.npz")):
    z = np.load(p, allow_pickle=True)
    ys, ps = song_level(z['gidx'], z['logits'])
    for g in np.unique(ys):
        per.setdefault(int(g), []).append(float((ps[ys == g] == g).mean()))
if per:
    pg = pd.DataFrame({'genre': [ID2GENRE[g] for g in sorted(per)],
                       'song_acc_mean': [np.mean(per[g]) for g in sorted(per)],
                       'folds_present': [len(per[g]) for g in sorted(per)]}).sort_values('song_acc_mean')
    print(pg.round(3).to_string(index=False))
    pg.to_csv(f"{RESULTS}/per_genre_R4_resnet18.csv", index=False)
print("\nAll artifacts saved to", RESULTS)

## After the full grid finishes, send me
1. The **summary table** (Cell 4 printout, all five regimes × both models).
2. The **collapse-curve figure**.
3. The **per-genre table** — I want to check the Tamang Selo prediction (22 artists ⇒ smallest R2→R3 drop) and the small-genre volatility.
4. Any run whose `val_macro_f1` looks broken (≈ chance) — name it and I'll diagnose.

Everything (checkpoints, per-segment logits, metrics) is persisted under `processed/results2A/`, so
Phase 2B (probes + ablations) and 2C (WLPA + the four playback singers) will reuse these runs
without retraining.

**Sanity anchors while it runs:** R1a should reach ≳0.95 within a few epochs; R3 val-F1 will be much
lower and noisier across folds — that is the expected behaviour, not a bug.
